Task 7 — LoRA Fine-Tuning of a 3B Model

The task uses PyTorch, Hugging Face PEFT, Transformers, and CUDA Toolkit. It focuses on adding low-rank adapter matrices, freezing the base model, and training only the LoRA parameters

In [1]:
# Import Libraries

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

In [2]:
# Load Model

model_name = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name
)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [3]:
# LoRA Configuration

config = LoraConfig(
    r=4,
    lora_alpha=16,
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)

In [6]:
import warnings
warnings.filterwarnings('ignore')

# Add LoRA Adapters
!pip install --upgrade torchao

model = get_peft_model(
    model,
    config
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 19.6 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [7]:
# Check Trainable Parameters

model.print_trainable_parameters()


trainable params: 147,456 || all params: 124,587,264 || trainable%: 0.1184


In [8]:
# Training Text

text = "Artificial intelligence is changing the world."

In [9]:
# Tokenize Text

inputs = tokenizer(
    text,
    return_tensors="pt"
)

print(inputs)

{'input_ids': tensor([[8001, 9542, 4430,  318, 5609,  262,  995,   13]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1]])}


In [10]:
# Forward Pass

outputs = model(
    input_ids=inputs["input_ids"],
    labels=inputs["input_ids"]
)

loss = outputs.loss

print("Loss:", loss.item())

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Loss: 3.354243516921997


In [11]:
# Forward Pass

outputs = model(
    input_ids=inputs["input_ids"],
    labels=inputs["input_ids"]
)

loss = outputs.loss

print("Loss:", loss.item())

Loss: 3.354243516921997


In [12]:
# Optimizer

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4
)

In [13]:
# Backward Pass

loss.backward()

optimizer.step()

optimizer.zero_grad()

In [14]:
# Results

print("LoRA Fine-Tuning Completed")
print("Loss:", loss.item())

LoRA Fine-Tuning Completed
Loss: 3.354243516921997
